# Apriori Algorithm Implementation Assignment

### Objective:
You will implement the **Apriori algorithm** from scratch (i.e., without using any libraries like `mlxtend`) to find frequent itemsets and generate association rules.

### Dataset:
Use the [Online Retail Dataset](https://www.kaggle.com/datasets/vijayuv/onlineretail) from Kaggle. You can filter it for a specific country (e.g., `United Kingdom`) and time range to reduce size if needed.

---

## Step 1: Data Preprocessing

- Load the dataset
- Remove rows with missing values
- Filter out rows where `Quantity <= 0`
- Convert Data into Basket Format

👉 **Implement code below**

In [ ]:
import pandas as pd
import numpy as np
from itertools import combinations

# Load and preprocess the dataset
df = pd.read_csv('OnlineRetail.csv', encoding='ISO-8859-1')

# Data preprocessing
print(f"Original dataset shape: {df.shape}")

# Remove rows with missing values
df = df.dropna()
print(f"After removing missing values: {df.shape}")

# Filter out rows where Quantity <= 0
df = df[df['Quantity'] > 0]
print(f"After filtering positive quantities: {df.shape}")

# Filter for United Kingdom to reduce dataset size
df = df[df['Country'] == 'United Kingdom']
print(f"After filtering for UK: {df.shape}")

# Convert to basket format
# Group by InvoiceNo and Description, sum quantities, and create binary matrix
basket = df.groupby(['InvoiceNo', 'Description'])['Quantity'].sum().unstack().reset_index().fillna(0).set_index('InvoiceNo')

# Convert to binary (1 if item was purchased, 0 otherwise)
basket = basket.applymap(lambda x: 1 if x > 0 else 0)

print(f"Basket matrix shape: {basket.shape}")
print(f"Number of transactions: {len(basket)}")
print(f"Number of unique items: {len(basket.columns)}")

# Convert basket to list of transactions (sets of items)
transactions = []
for _, row in basket.iterrows():
    transaction = set([item for item in basket.columns if row[item] == 1])
    transactions.append(transaction)

print(f"Sample transaction: {list(list(transactions[0])[:5])}") # Show first 5 items of first transaction

Original dataset shape: (541909, 8)
After removing missing values: (406829, 8)
After filtering positive quantities: (397924, 8)
After filtering for UK: (354345, 8)


C:\Users\ASUS\AppData\Local\Temp\ipykernel_2660\3246517409.py:28: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  basket = basket.applymap(lambda x: 1 if x > 0 else 0)


Basket matrix shape: (16649, 3844)
Number of transactions: 16649
Number of unique items: 3844


## Step 2: Implement Apriori Algorithm
Step-by-Step Procedure:
1. Generate Frequent 1-Itemsets
Count the frequency (support) of each individual item in the dataset.
Keep only those with support ≥ min_support.
→ Result is L1 (frequent 1-itemsets)
2. Iterative Candidate Generation (k = 2 to n)
While L(k-1) is not empty:
a. Candidate Generation

Generate candidate itemsets Ck of size k from L(k-1) using the Apriori property:
Any (k-itemset) is only frequent if all of its (k−1)-subsets are frequent.
b. Prune Candidates
Eliminate candidates that have any (k−1)-subset not in L(k-1).
c. Count Support
For each transaction, count how many times each candidate in Ck appears.
d. Generate Frequent Itemsets
Form Lk by keeping candidates from Ck that meet the min_support.
Repeat until Lk becomes empty.
Implement the following functions:
1. `get_frequent_itemsets(transactions, min_support)` - Returns frequent itemsets and their support
2. `generate_candidates(prev_frequent_itemsets, k)` - Generates candidate itemsets of length `k`
3. `calculate_support(transactions, candidates)` - Calculates the support count for each candidate

**Write reusable functions** for each part of the algorithm.

In [9]:
# Implement apriori functions below

def calculate_support(transactions, candidates):
    """Calculate support count for each candidate itemset"""
    support_counts = {}
    
    for candidate in candidates:
        count = 0
        for transaction in transactions:
            if candidate.issubset(transaction):
                count += 1
        support_counts[candidate] = count
    
    return support_counts

def generate_candidates(prev_frequent_itemsets, k):
    """Generate candidate itemsets of length k from frequent itemsets of length k-1"""
    candidates = set()
    prev_itemsets_list = list(prev_frequent_itemsets)
    
    # Generate candidates by joining itemsets that differ by only one item
    for i in range(len(prev_itemsets_list)):
        for j in range(i + 1, len(prev_itemsets_list)):
            itemset1 = prev_itemsets_list[i]
            itemset2 = prev_itemsets_list[j]
            
            # Union of two itemsets
            union = itemset1.union(itemset2)
            
            # Only add if the union has exactly k items
            if len(union) == k:
                # Check if all k-1 subsets are frequent (pruning step)
                valid = True
                for subset in combinations(union, k-1):
                    if frozenset(subset) not in prev_frequent_itemsets:
                        valid = False
                        break
                
                if valid:
                    candidates.add(frozenset(union))
    
    return candidates

def get_frequent_itemsets(transactions, min_support):
    """Main Apriori algorithm implementation"""
    num_transactions = len(transactions)
    min_support_count = int(min_support * num_transactions)
    
    # Get all unique items
    all_items = set()
    for transaction in transactions:
        all_items.update(transaction)
    
    # Generate frequent 1-itemsets
    candidates_1 = [frozenset([item]) for item in all_items]
    support_counts_1 = calculate_support(transactions, candidates_1)
    
    frequent_1 = set()
    for itemset, count in support_counts_1.items():
        if count >= min_support_count:
            frequent_1.add(itemset)
    
    # Dictionary to store all frequent itemsets with their support
    all_frequent_itemsets = {}
    for itemset in frequent_1:
        all_frequent_itemsets[itemset] = support_counts_1[itemset] / num_transactions
    
    current_frequent = frequent_1
    k = 2
    
    # Generate frequent itemsets of length k
    while current_frequent and k <= 4:  # Limit to avoid too long computation
        print(f"Generating frequent {k}-itemsets...")
        
        # Generate candidates
        candidates = generate_candidates(current_frequent, k)
        
        if not candidates:
            break
        
        # Calculate support for candidates
        support_counts = calculate_support(transactions, candidates)
        
        # Filter frequent itemsets
        current_frequent = set()
        for itemset, count in support_counts.items():
            if count >= min_support_count:
                current_frequent.add(itemset)
                all_frequent_itemsets[itemset] = count / num_transactions
        
        print(f"Found {len(current_frequent)} frequent {k}-itemsets")
        k += 1
    
    return all_frequent_itemsets

# Test the implementation
min_support = 0.01  # 1% minimum support
print(f"\nRunning Apriori with minimum support: {min_support}")
frequent_itemsets = get_frequent_itemsets(transactions, min_support)

print(f"\nTotal frequent itemsets found: {len(frequent_itemsets)}")

# Show some examples
print("\nTop 10 frequent itemsets by support:")
sorted_itemsets = sorted(frequent_itemsets.items(), key=lambda x: x[1], reverse=True)
for i, (itemset, support) in enumerate(sorted_itemsets[:10]):
    items_str = ', '.join(list(itemset)[:3])  # Show first 3 items to keep output readable
    if len(itemset) > 3:
        items_str += f" (and {len(itemset)-3} more)"
    print(f"{i+1}. {items_str} - Support: {support:.4f}")


Running Apriori with minimum support: 0.01
Generating frequent 2-itemsets...
Found 304 frequent 2-itemsets
Generating frequent 3-itemsets...
Found 60 frequent 3-itemsets
Generating frequent 4-itemsets...
Found 3 frequent 4-itemsets

Total frequent itemsets found: 984

Top 10 frequent itemsets by support:
1. WHITE HANGING HEART T-LIGHT HOLDER - Support: 0.1132
2. JUMBO BAG RED RETROSPOT - Support: 0.0869
3. REGENCY CAKESTAND 3 TIER - Support: 0.0847
4. ASSORTED COLOUR BIRD ORNAMENT - Support: 0.0781
5. PARTY BUNTING - Support: 0.0775
6. LUNCH BAG RED RETROSPOT - Support: 0.0673
7. SET OF 3 CAKE TINS PANTRY DESIGN  - Support: 0.0605
8. LUNCH BAG  BLACK SKULL. - Support: 0.0598
9. PAPER CHAIN KIT 50'S CHRISTMAS  - Support: 0.0568
10. NATURAL SLATE HEART CHALKBOARD  - Support: 0.0563


## Step 3: Generate Association Rules

- Use frequent itemsets to generate association rules
- For each rule `A => B`, calculate:
  - **Support**
  - **Confidence**
- Only return rules that meet a minimum confidence threshold (e.g., 0.5)

👉 **Implement rule generation function below**

In [11]:
# Function to generate rules from frequent itemsets

def generate_rules(frequent_itemsets, min_confidence):
    """Generate association rules from frequent itemsets"""
    rules = []
    
    # Only consider itemsets with more than 1 item
    for itemset, support in frequent_itemsets.items():
        if len(itemset) < 2:
            continue
        
        # Generate all possible non-empty proper subsets as antecedents
        items = list(itemset)
        for r in range(1, len(items)):
            for antecedent_items in combinations(items, r):
                antecedent = frozenset(antecedent_items)
                consequent = itemset - antecedent
                
                if len(consequent) == 0:
                    continue
                
                # Calculate confidence: support(itemset) / support(antecedent)
                if antecedent in frequent_itemsets:
                    antecedent_support = frequent_itemsets[antecedent]
                    confidence = support / antecedent_support
                    
                    if confidence >= min_confidence:
                        # Calculate lift: confidence / support(consequent)
                        consequent_support = frequent_itemsets.get(consequent, 0)
                        lift = confidence / consequent_support if consequent_support > 0 else 0
                        
                        rules.append({
                            'antecedent': antecedent,
                            'consequent': consequent,
                            'support': support,
                            'confidence': confidence,
                            'lift': lift
                        })
    
    return rules

# Generate association rules
min_confidence = 0.5  # 50% minimum confidence
print(f"\nGenerating association rules with minimum confidence: {min_confidence}")

rules = generate_rules(frequent_itemsets, min_confidence)
print(f"Generated {len(rules)} association rules")


Generating association rules with minimum confidence: 0.5
Generated 278 association rules


## Step 4: Output and Visualize

- Print top 10 frequent itemsets
- Print top 10 association rules (by confidence or lift)

👉 **Output results below**

In [ ]:
# Output the final results
all_items = set()
for transaction in transactions:
    all_items.update(transaction)

print("\n" + "="*80)
print("FINAL RESULTS - TOP 10 FREQUENT ITEMSETS")
print("="*80)

# Sort itemsets by support (descending)
sorted_itemsets = sorted(frequent_itemsets.items(), key=lambda x: x[1], reverse=True)

for i, (itemset, support) in enumerate(sorted_itemsets[:10]):
    items_list = list(itemset)
    items_str = ', '.join(items_list[:2])  # Show first 2 items
    if len(itemset) > 2:
        items_str += f" (+ {len(itemset)-2} more items)"
    print(f"{i+1:2d}. [{items_str}]"
        f" | Support: {support:.4f} | Size: {len(itemset)}")

print("\n" + "="*80)
print("FINAL RESULTS - TOP 10 ASSOCIATION RULES")
print("="*80)

if rules:
    # Sort rules by confidence (descending)
    sorted_rules = sorted(rules, key=lambda x: x['confidence'], reverse=True)
    
    for i, rule in enumerate(sorted_rules[:10]):
        antecedent_str = ', '.join(list(rule['antecedent'])[:2])
        if len(rule['antecedent']) > 2:
            antecedent_str += f" (+ {len(rule['antecedent'])-2} more)"
        
        consequent_str = ', '.join(list(rule['consequent'])[:2])
        if len(rule['consequent']) > 2:
            consequent_str += f" (+ {len(rule['consequent'])-2} more)"
        
        print(f"{i+1:2d}. [{antecedent_str}] => [{consequent_str}]")
        print(f"    Support: {rule['support']:.4f} | "
            f"Confidence: {rule['confidence']:.4f} | "
            f"Lift: {rule['lift']:.4f}")
        print()
else:
    print("No association rules found with the given minimum confidence.")

# Summary statistics
print("\n" + "="*80)
print("SUMMARY STATISTICS")
print("="*80)

itemset_sizes = {}
for itemset in frequent_itemsets.keys():
    size = len(itemset)
    itemset_sizes[size] = itemset_sizes.get(size, 0) + 1

print(f"Total transactions processed: {len(transactions)}")
print(f"Total unique items: {len(all_items)}")
print(f"Minimum support threshold: {min_support}")
print(f"Minimum confidence threshold: {min_confidence}")
print(f"Total frequent itemsets: {len(frequent_itemsets)}")

print("Frequent itemsets by size:")
for size, count in sorted(itemset_sizes.items()):
    print(f"  {size}-itemsets: {count}")

print(f"Total association rules: {len(rules) if rules else 0}")

if rules:
    avg_confidence = sum(rule['confidence'] for rule in rules) / len(rules)
    avg_lift = sum(rule['lift'] for rule in rules) / len(rules)
    print(f"Average confidence: {avg_confidence:.4f}")
    print(f"Average lift: {avg_lift:.4f}")


FINAL RESULTS - TOP 10 FREQUENT ITEMSETS
 1. [WHITE HANGING HEART T-LIGHT HOLDER] | Support: 0.1132 | Size: 1
 2. [JUMBO BAG RED RETROSPOT] | Support: 0.0869 | Size: 1
 3. [REGENCY CAKESTAND 3 TIER] | Support: 0.0847 | Size: 1
 4. [ASSORTED COLOUR BIRD ORNAMENT] | Support: 0.0781 | Size: 1
 5. [PARTY BUNTING] | Support: 0.0775 | Size: 1
 6. [LUNCH BAG RED RETROSPOT] | Support: 0.0673 | Size: 1
 7. [SET OF 3 CAKE TINS PANTRY DESIGN ] | Support: 0.0605 | Size: 1
 8. [LUNCH BAG  BLACK SKULL.] | Support: 0.0598 | Size: 1
 9. [PAPER CHAIN KIT 50'S CHRISTMAS ] | Support: 0.0568 | Size: 1
10. [NATURAL SLATE HEART CHALKBOARD ] | Support: 0.0563 | Size: 1

FINAL RESULTS - TOP 10 ASSOCIATION RULES
 1. [HERB MARKER THYME] => [HERB MARKER ROSEMARY]
    Support: 0.0102 | Confidence: 0.9441 | Lift: 86.8447

 2. [HERB MARKER ROSEMARY] => [HERB MARKER THYME]
    Support: 0.0102 | Confidence: 0.9337 | Lift: 86.8447

 3. [HERB MARKER PARSLEY] => [HERB MARKER ROSEMARY]
    Support: 0.0100 | Confidence: 